In [ ]:
# ─────────────────────────────────────────────
# PART 1: Install & Imports
# ─────────────────────────────────────────────
!pip install open3d plotly huggingface_hub scikit-image numpy scipy wandb -q

import open3d as o3d
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.ndimage import distance_transform_edt
from skimage.measure import marching_cubes
import wandb
import os
import zipfile
from huggingface_hub import hf_hub_download

SEED = 42
np.random.seed(SEED)
print("All imports done.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 447.7/447.7 MB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 100.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.8/139.8 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 82.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 78.2 MB/s eta 0:00:00
All imports done.


In [ ]:
# ─────────────────────────────────────────────
# PART 2: Download & Extract Dataset
# ─────────────────────────────────────────────
zip_path = hf_hub_download(
    repo_id="BGLab/AgriField3D",
    filename="datasets/FielGrwon_ZeaMays_RawPCD_10k.zip",
    repo_type="dataset",
    local_dir="./data"
)

extract_dir = "./data/RawPCD_10k"
os.makedirs(extract_dir, exist_ok=True)
with zipfile.ZipFile(zip_path, 'r') as zf:
    zf.extractall(extract_dir)

ply_files = sorted([
    os.path.join(root, f)
    for root, _, files in os.walk(extract_dir)
    for f in files if f.endswith('.ply')
])
print(f"Total plants found: {len(ply_files)}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


datasets/FielGrwon_ZeaMays_RawPCD_10k.zi(…):   0%|          | 0.00/218M [00:00<?, ?B/s]

Total plants found: 1045


In [ ]:
# ─────────────────────────────────────────────
# PART 3: Load & Normalise One Plant
# ─────────────────────────────────────────────
PLANT_IDX  = 0
NUM_POINTS = 4096
VOXEL_RES  = 64

# ── Load raw point cloud ──────────────────────────────────────────────
pcd = o3d.io.read_point_cloud(ply_files[PLANT_IDX])
pts = np.asarray(pcd.points, dtype=np.float32)
print(f"Raw points      : {pts.shape}")

# ── Subsample ─────────────────────────────────────────────────────────
N   = pts.shape[0]
idx = (np.random.choice(N, NUM_POINTS, replace=False)
       if N >= NUM_POINTS
       else np.random.choice(N, NUM_POINTS, replace=True))
pts = pts[idx]

# ── Normalise to [-1, 1]^3 ────────────────────────────────────────────
centroid = pts.mean(axis=0)
pts     -= centroid
scale    = np.linalg.norm(pts, axis=1).max() + 1e-8
pts     /= scale

print(f"After sampling  : {pts.shape}")
print(f"Value range     : [{pts.min():.3f}, {pts.max():.3f}]")

Raw points      : (10000, 3)
After sampling  : (4096, 3)
Value range     : [-0.735, 0.991]


In [ ]:
# ─────────────────────────────────────────────
# PART 4: Compute All 4 Representations
# ─────────────────────────────────────────────

# ════════════════════════════════════════════
# 1. POINT CLOUD
#    Raw (x,y,z) coordinates — simplest form
# ════════════════════════════════════════════
point_cloud = pts.copy()   # [N, 3]
print(f"[1] Point Cloud : {point_cloud.shape}")


# ════════════════════════════════════════════
# 2. MESH
#    Triangle mesh via Ball Pivoting Algorithm
#    on the full original point cloud.
#    Vertices + triangle faces define the
#    continuous surface.
# ════════════════════════════════════════════
pcd_full = o3d.io.read_point_cloud(ply_files[PLANT_IDX])
pcd_full.estimate_normals(
    search_param=o3d.geometry.KDTreeSearchParamHybrid(
        radius=0.1, max_nn=30))

distances = pcd_full.compute_nearest_neighbor_distance()
avg_dist  = np.mean(distances)
radius    = 3 * avg_dist

mesh_o3d  = o3d.geometry.TriangleMesh\
               .create_from_point_cloud_ball_pivoting(
                   pcd_full,
                   o3d.utility.DoubleVector(
                       [radius, radius * 2]))
mesh_o3d.compute_vertex_normals()

mesh_verts = np.asarray(mesh_o3d.vertices,  dtype=np.float32)
mesh_faces = np.asarray(mesh_o3d.triangles, dtype=np.int32)

# Normalise mesh vertices same as point cloud
mesh_verts -= centroid
mesh_verts /= scale

print(f"[2] Mesh        : {mesh_verts.shape[0]} verts, "
      f"{mesh_faces.shape[0]} faces")


# ════════════════════════════════════════════
# 3. VOXEL GRID
#    Discretise point cloud into a binary
#    occupancy grid [R, R, R].
#    1 = occupied, 0 = empty.
# ════════════════════════════════════════════
def points_to_voxels(pts, resolution):
    """
    Map normalised points [-1,1]^3
    → binary occupancy grid [R,R,R].
    """
    grid_pts  = ((pts + 1.0) / 2.0 * (resolution - 1)
                ).clip(0, resolution - 1).astype(int)
    occupancy = np.zeros(
        (resolution, resolution, resolution),
        dtype=np.uint8)
    occupancy[grid_pts[:,0],
              grid_pts[:,1],
              grid_pts[:,2]] = 1
    return occupancy

voxel_grid = points_to_voxels(pts, VOXEL_RES)
print(f"[3] Voxel Grid  : {voxel_grid.shape} | "
      f"Occupied: {voxel_grid.sum()} / "
      f"{VOXEL_RES**3} voxels "
      f"({100*voxel_grid.mean():.2f}%)")


# ════════════════════════════════════════════
# 4. SIGNED DISTANCE FIELD (SDF)
#    For each voxel compute signed distance
#    to nearest surface point.
#    Negative = inside, Positive = outside.
#    Normalised + truncated to [-0.1, 0.1].
# ════════════════════════════════════════════
def voxels_to_sdf(occupancy, resolution, trunc=0.1):
    """
    Compute truncated SDF from occupancy grid.
    Uses scipy distance transform — fast & exact.
    """
    dist_out = distance_transform_edt(
                   1 - occupancy).astype(np.float32)
    dist_in  = distance_transform_edt(
                   occupancy).astype(np.float32)
    sdf      = dist_out - dist_in

    # Convert from voxels → metric (grid spans [-1,1])
    voxel_size = 2.0 / resolution
    sdf       *= voxel_size

    # Normalise to [-1,1]
    abs_max    = np.abs(sdf).max()
    sdf       /= (abs_max + 1e-8)

    # Truncate
    sdf        = np.clip(sdf, -trunc, trunc)
    return sdf

sdf_grid = voxels_to_sdf(voxel_grid, VOXEL_RES)
print(f"[4] SDF Grid    : {sdf_grid.shape} | "
      f"Range [{sdf_grid.min():.3f}, "
      f"{sdf_grid.max():.3f}]")

[1] Point Cloud : (4096, 3)
[2] Mesh        : 10000 verts, 10752 faces
[3] Voxel Grid  : (64, 64, 64) | Occupied: 809 / 262144 voxels (0.31%)
[4] SDF Grid    : (64, 64, 64) | Range [-0.033, 0.100]


In [ ]:
# ─────────────────────────────────────────────
# PART 5: Visualize All 4 Representations
# ─────────────────────────────────────────────

# ════════════════════════════════════════════
# VIZ 1 — Point Cloud
# ════════════════════════════════════════════
fig_pcd = go.Figure(go.Scatter3d(
    x=point_cloud[:,0],
    y=point_cloud[:,1],
    z=point_cloud[:,2],
    mode='markers',
    marker=dict(
        size=1.5,
        color=point_cloud[:,2],      # colour by height
        colorscale='Viridis',
        colorbar=dict(title='Z'),
        showscale=True
    ),
    name='Point Cloud'
))
fig_pcd.update_layout(
    title    = "Representation 1 — Point Cloud",
    template = "plotly_dark",
    scene    = dict(
        aspectmode='data',
        xaxis_title='X',
        yaxis_title='Y',
        zaxis_title='Z'
    ),
    height = 600
)
fig_pcd.show()


# ════════════════════════════════════════════
# VIZ 2 — Mesh
# ════════════════════════════════════════════
fig_mesh = go.Figure(go.Mesh3d(
    x=mesh_verts[:,0],
    y=mesh_verts[:,1],
    z=mesh_verts[:,2],
    i=mesh_faces[:,0],
    j=mesh_faces[:,1],
    k=mesh_faces[:,2],
    intensity=mesh_verts[:,2],
    colorscale='Teal',
    opacity=0.85,
    showscale=True,
    colorbar=dict(title='Z'),
    name='Mesh'
))
fig_mesh.update_layout(
    title    = "Representation 2 — Triangle Mesh",
    template = "plotly_dark",
    scene    = dict(
        aspectmode='data',
        xaxis_title='X',
        yaxis_title='Y',
        zaxis_title='Z'
    ),
    height = 600
)
fig_mesh.show()


# ════════════════════════════════════════════
# VIZ 3 — Voxel Grid
# 3a. 3D scatter of occupied voxels
# 3b. Cross-section slices (XY, XZ, YZ)
# ════════════════════════════════════════════

# 3a — 3D occupied voxels
occ_idx   = np.argwhere(voxel_grid > 0)          # [K, 3]
# Map back to [-1, 1]
occ_coords = occ_idx / (VOXEL_RES - 1) * 2 - 1  # [K, 3]

fig_vox3d = go.Figure(go.Scatter3d(
    x=occ_coords[:,0],
    y=occ_coords[:,1],
    z=occ_coords[:,2],
    mode='markers',
    marker=dict(
        size=2,
        color=occ_coords[:,2],
        colorscale='Plasma',
        showscale=True,
        colorbar=dict(title='Z'),
        opacity=0.6
    ),
    name='Occupied Voxels'
))
fig_vox3d.update_layout(
    title    = "Representation 3a — Voxel Grid (3D)",
    template = "plotly_dark",
    scene    = dict(
        aspectmode='data',
        xaxis_title='X',
        yaxis_title='Y',
        zaxis_title='Z'
    ),
    height = 600
)
fig_vox3d.show()

# 3b — Cross-section slices
mid = VOXEL_RES // 2
fig_vox_slices = make_subplots(
    rows=1, cols=3,
    subplot_titles=[
        f"XY Slice (z={mid})",
        f"XZ Slice (y={mid})",
        f"YZ Slice (x={mid})"
    ]
)
slices = [
    voxel_grid[:, :, mid],
    voxel_grid[:, mid, :],
    voxel_grid[mid, :, :]
]
for col, sl in enumerate(slices, start=1):
    fig_vox_slices.add_trace(
        go.Heatmap(
            z=sl.T,
            colorscale='Greys',
            showscale=False,
            zmin=0, zmax=1
        ),
        row=1, col=col
    )
fig_vox_slices.update_layout(
    title    = "Representation 3b — Voxel Cross-Sections",
    template = "plotly_dark",
    height   = 350
)
fig_vox_slices.show()


# ════════════════════════════════════════════
# VIZ 4 — SDF
# 4a. Isosurface at SDF = 0 (the actual surface)
# 4b. Cross-section heatmaps (RdBu — red=outside,
#     blue=inside)
# 4c. SDF value distribution histogram
# ════════════════════════════════════════════

# 4a — Isosurface via marching cubes
try:
    verts_mc, faces_mc, _, _ = marching_cubes(
        sdf_grid, level=0.0)
    # Normalise back to [-1,1]
    verts_mc = verts_mc / (VOXEL_RES - 1) * 2 - 1

    fig_sdf_surf = go.Figure(go.Mesh3d(
        x=verts_mc[:,0],
        y=verts_mc[:,1],
        z=verts_mc[:,2],
        i=faces_mc[:,0],
        j=faces_mc[:,1],
        k=faces_mc[:,2],
        intensity=verts_mc[:,2],
        colorscale='RdBu',
        opacity=0.8,
        showscale=True,
        colorbar=dict(title='Z'),
        name='SDF Surface'
    ))
    fig_sdf_surf.update_layout(
        title    = "Representation 4a — SDF Isosurface (level=0)",
        template = "plotly_dark",
        scene    = dict(
            aspectmode='data',
            xaxis_title='X',
            yaxis_title='Y',
            zaxis_title='Z'
        ),
        height = 600
    )
    fig_sdf_surf.show()
except Exception as e:
    print(f"Marching cubes failed: {e}")

# 4b — SDF cross-section heatmaps
fig_sdf_slices = make_subplots(
    rows=1, cols=3,
    subplot_titles=[
        f"SDF XY Slice (z={mid})",
        f"SDF XZ Slice (y={mid})",
        f"SDF YZ Slice (x={mid})"
    ]
)
sdf_slices = [
    sdf_grid[:, :, mid],
    sdf_grid[:, mid, :],
    sdf_grid[mid, :, :]
]
for col, sl in enumerate(sdf_slices, start=1):
    fig_sdf_slices.add_trace(
        go.Heatmap(
            z=sl.T,
            colorscale='RdBu',
            zmid=0,            # centre colorscale at 0
            showscale=(col==3),
            colorbar=dict(title='SDF')
        ),
        row=1, col=col
    )
fig_sdf_slices.update_layout(
    title    = "Representation 4b — SDF Cross-Sections "
               "(Red=Outside, Blue=Inside)",
    template = "plotly_dark",
    height   = 380
)
fig_sdf_slices.show()

# 4c — SDF value distribution
fig_sdf_hist = go.Figure(go.Histogram(
    x=sdf_grid.flatten(),
    nbinsx=80,
    marker_color='#636EFA',
    opacity=0.8,
    name='SDF values'
))
fig_sdf_hist.add_vline(
    x=0, line_dash="dash",
    line_color="red",
    annotation_text="Surface (SDF=0)",
    annotation_position="top right"
)
fig_sdf_hist.update_layout(
    title       = "Representation 4c — SDF Value Distribution",
    xaxis_title = "SDF Value",
    yaxis_title = "Voxel Count",
    template    = "plotly_dark",
    height      = 400
)
fig_sdf_hist.show()

<

In [ ]:
# ─────────────────────────────────────────────
# PART 6: Side-by-Side + Log to W&B
# ─────────────────────────────────────────────
wandb.init(
    project = "maize-representations",
    name    = "all-4-representations"
)

# ── Combined 4-panel comparison ───────────────────────────────────────
fig_all = make_subplots(
    rows=1, cols=4,
    specs=[[{"type":"scatter3d"},
            {"type":"mesh3d"},
            {"type":"scatter3d"},
            {"type":"mesh3d"}]],
    subplot_titles=[
        "Point Cloud",
        "Mesh",
        "Voxels",
        "SDF Surface"
    ],
    horizontal_spacing=0.02
)

# Panel 1 — Point Cloud
fig_all.add_trace(go.Scatter3d(
    x=point_cloud[:,0],
    y=point_cloud[:,1],
    z=point_cloud[:,2],
    mode='markers',
    marker=dict(size=1, color=point_cloud[:,2],
                colorscale='Viridis', showscale=False),
    showlegend=False
), row=1, col=1)

# Panel 2 — Mesh
fig_all.add_trace(go.Mesh3d(
    x=mesh_verts[:,0],
    y=mesh_verts[:,1],
    z=mesh_verts[:,2],
    i=mesh_faces[:,0],
    j=mesh_faces[:,1],
    k=mesh_faces[:,2],
    color='lightblue',
    opacity=0.8,
    showlegend=False
), row=1, col=2)

# Panel 3 — Voxels
fig_all.add_trace(go.Scatter3d(
    x=occ_coords[:,0],
    y=occ_coords[:,1],
    z=occ_coords[:,2],
    mode='markers',
    marker=dict(size=1.5, color=occ_coords[:,2],
                colorscale='Plasma', showscale=False,
                opacity=0.5),
    showlegend=False
), row=1, col=3)

# Panel 4 — SDF surface
if 'verts_mc' in dir() and verts_mc is not None:
    fig_all.add_trace(go.Mesh3d(
        x=verts_mc[:,0],
        y=verts_mc[:,1],
        z=verts_mc[:,2],
        i=faces_mc[:,0],
        j=faces_mc[:,1],
        k=faces_mc[:,2],
        color='lightsalmon',
        opacity=0.8,
        showlegend=False
    ), row=1, col=4)

fig_all.update_layout(
    title    = ("Maize Plant — All 4 Geometric Representations "
                "Side by Side"),
    template = "plotly_dark",
    height   = 600,
    margin   = dict(l=10, r=10, t=60, b=10)
)
fig_all.show()

# ── Log everything to W&B ─────────────────────────────────────────────
wandb.log({
    "point_cloud"         : wandb.Plotly(fig_pcd),
    "mesh"                : wandb.Plotly(fig_mesh),
    "voxel_3d"            : wandb.Plotly(fig_vox3d),
    "voxel_slices"        : wandb.Plotly(fig_vox_slices),
    "sdf_surface"         : wandb.Plotly(fig_sdf_surf),
    "sdf_slices"          : wandb.Plotly(fig_sdf_slices),
    "sdf_distribution"    : wandb.Plotly(fig_sdf_hist),
    "all_representations" : wandb.Plotly(fig_all),
})

# ── Log 3D point cloud as W&B Object3D ───────────────────────────────
rgb = np.tile([0, 120, 255], (len(point_cloud), 1))
wandb.log({
    "point_cloud_3d": wandb.Object3D(
        np.hstack([point_cloud, rgb]))
})

# ── Stats summary table ────────────────────────────────────────────────
wandb.log({
    "representation_summary": wandb.Table(
        columns=["Representation", "Description",
                 "Size", "Memory (approx)"],
        data=[
            ["Point Cloud",
             "Raw (x,y,z) coordinates",
             f"{NUM_POINTS} points",
             f"{NUM_POINTS*3*4/1024:.1f} KB"],
            ["Mesh",
             "Vertices + triangle faces",
             f"{mesh_verts.shape[0]} verts, "
             f"{mesh_faces.shape[0]} faces",
             f"{(mesh_verts.nbytes+mesh_faces.nbytes)/1024:.1f} KB"],
            ["Voxel Grid",
             "Binary occupancy [R,R,R]",
             f"{VOXEL_RES}^3 = {VOXEL_RES**3} voxels",
             f"{VOXEL_RES**3/1024:.1f} KB"],
            ["SDF",
             "Signed distance [-1,1], truncated",
             f"{VOXEL_RES}^3 = {VOXEL_RES**3} values",
             f"{VOXEL_RES**3*4/1024:.1f} KB"],
        ]
    )
})

wandb.finish()
print("All representations visualized and logged.")

All representations visualized and logged.
